# Dataset Join: Archetypometrics × IMDB Title Basics

This notebook joins two datasets:
- **archetypometricsdata2000.mat** — 2,000 fictional characters from 341 stories, rated on 464 bipolar personality trait scales with SVD-based archetype assignments
- **title.basics.tsv** — IMDB title metadata (12M+ titles) including type, year, runtime, and genres

**Join strategy:** Extract character → story mappings from the `.mat` file, then match story names against IMDB `primaryTitle` using exact matching first, with fuzzy fallback for unmatched stories.

In [ ]:
import scipy.io
import pandas as pd
import numpy as np
from difflib import get_close_matches

mat = scipy.io.loadmat('archetypometricsdata2000.mat', simplify_cells=True)
print("Top-level keys:", [k for k in mat.keys() if not k.startswith('__')])

Top-level keys: ['data_archetype_space', 'data_characters', 'data_characters_latex', 'data_raw', 'data_stories', 'data_traits', 'data_traits_latex', 'framework_archetypes']


## Dataset 1 Summary: Archetypometrics

In [12]:
# --- Extract characters ---
chars_raw = mat['data_characters']
char_names = list(chars_raw['character_typenames'])
char_story_pairs = list(chars_raw['character_story_typenames'])  # "CharName/StoryName"

# Parse story name from the "CharName/StoryName" field
story_names_per_char = [pair.split('/', 1)[1] if '/' in pair else pair for pair in char_story_pairs]

# --- Extract stories (all 341 unique storyverses) ---
stories_raw = mat['data_stories']
all_story_names = list(stories_raw['storyverses'])

# --- Extract archetype assignments ---
archetype_space = mat['data_archetype_space']
print(archetype_space.keys())
primary_archetypes = list(archetype_space['character_archetypes_ordered'])

# --- Extract archetype ratio scores (max confidence across all 232 archetypes) ---
archetype_ratios = archetype_space['archetype_ratios']  # (2000, 232)

# --- Build character DataFrame ---
df_chars = pd.DataFrame({
    'character_name': char_names,
    'character_story_pair': char_story_pairs,
    'story_name': story_names_per_char,
    'primary_archetype': primary_archetypes,
    'max_archetype_ratio': archetype_ratios.max(axis=1),
})

print(f"Characters: {len(df_chars):,}")
print(f"Unique stories: {df_chars['story_name'].nunique()}")
print(f"Unique archetypes: {df_chars['primary_archetype'].nunique()}")
print()
df_chars.head(10)

dict_keys(['U', 'Sigma', 'V', 'maxtraitstrength', 'trait_alignment_cosines', 'trait_component_norms', 'trait_variance_explained', 'maxcharacterstrength', 'character_alignment_cosines', 'character_component_norms', 'character_variance_explained', 'archetypes', 'archetype_ratios', 'character_archetypes_ordered', 'character_archetypes_by_class'])
Characters: 2,000
Unique stories: 341
Unique archetypes: 409



,character_name,character_story_pair,story_name,primary_archetype,max_archetype_ratio
0,Applejack,Applejack/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Angel-Brute-Hero,4.879310
1,Twilight Sparkle,Twilight Sparkle/My Little Pony: Friendship Is...,My Little Pony: Friendship Is Magic,Diva-Angel-Hero,10.590790
2,Rarity,Rarity/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Sophisticate-Diva-Hero,12.078822
3,Pinkie Pie,Pinkie Pie/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Adventurer,11.830696
4,Spike,Spike/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Angel-Outcast-Adventurer,11.792687
5,Princess Celestia,Princess Celestia/My Little Pony: Friendship I...,My Little Pony: Friendship Is Magic,Sophisticate-Angel-Hero,23.942371
6,Leroy Jethro Gibbs,Leroy Jethro Gibbs/NCIS,NCIS,Hero,7.959240
7,Anthony DiNozzo,Anthony DiNozzo/NCIS,NCIS,Hero-Brute-Adventurer,8.321907
8,Abby Sciuto,Abby Sciuto/NCIS,NCIS,Geek-Hero-Adventurer,9.163252
9,Donald Mallard,Donald Mallard/NCIS,NCIS,Angel-Hero,10.309380


In [ ]:
traits = mat['data_traits']
U_matrix = archetype_space['U']
V_matrix = archetype_space['V']
raw_A = mat['data_raw']
print(U_matrix.shape)
print(traits.keys())
U_matrix_T = U_matrix.transpose()

(464, 464)
dict_keys(['trait_table', 'trait_flip_typenames', 'trait_typenames', 'urls'])


In [3]:
print("Archetype distribution (top 15):")
print(df_chars['primary_archetype'].value_counts().head(15))
print()
print("Sample stories:", all_story_names[:10])

Archetype distribution (top 15):
primary_archetype
Hero                   191
Angel-Hero              78
Adventurer              64
Hero-Demon              53
Demon                   45
Demon-Hero              40
Traditionalist-Hero     38
Hero-Angel              36
Adventurer-Hero         29
Angel                   27
Angel-Adventurer        27
Hero-Adventurer         22
Lone Wolf-Hero          22
Demon-Adventurer        19
Diva-Angel-Hero         19
Name: count, dtype: int64

Sample stories: ['(500) Days of Summer', '10 Things I Hate About You', '30 Rock', '8 Mile', 'A Nightmare on Elm Street', 'A Series of Unfortunate Events', 'A Star Is Born', 'A Tale of Two Cities', 'Adventures of Huckleberry Finn', 'After Life']


## Dataset 2 Summary: IMDB Title Basics

In [39]:
df_imdb = pd.read_csv(
    'title.basics.tsv',
    sep='\t',
    na_values='\\N',
    low_memory=False
)
# Cast numeric columns after load to avoid parse errors on mixed-type fields
for col in ['startYear', 'endYear', 'runtimeMinutes', 'isAdult']:
    df_imdb[col] = pd.to_numeric(df_imdb[col], errors='coerce')

print(f"Rows: {len(df_imdb):,}")
print(f"Columns: {list(df_imdb.columns)}")
print()
print("Title type distribution:")
print(df_imdb['titleType'].value_counts())
print()
df_imdb.head(5)

# get row where primaryTitle = 30 Rock
rock = df_imdb[df_imdb['primaryTitle'] == '30 Rock']

print(rock)

Rows: 12,390,593
Columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']

Title type distribution:
titleType
tvEpisode       9566844
short           1121351
movie            741591
video            323639
tvSeries         296793
tvMovie          154341
tvMiniSeries      69132
tvSpecial         57458
videoGame         48488
tvShort           10955
tvPilot               1
Name: count, dtype: int64

             tconst  titleType primaryTitle originalTitle  isAdult  startYear  \
476780    tt0496424   tvSeries      30 Rock       30 Rock        0     2006.0   
4155396  tt15869208  tvEpisode      30 Rock       30 Rock        0     2022.0   
4852139   tt1860375   tvSeries      30 Rock       30 Rock        0     1983.0   

         endYear  runtimeMinutes      genres  
476780    2013.0            22.0      Comedy  
4155396      NaN             NaN  Reality-TV  
4852139      NaN             NaN       Sport  


In [40]:
rock = df_imdb[df_imdb['primaryTitle'] == '30 Rock']

print(rock)

             tconst  titleType primaryTitle originalTitle  isAdult  startYear  \
476780    tt0496424   tvSeries      30 Rock       30 Rock        0     2006.0   
4155396  tt15869208  tvEpisode      30 Rock       30 Rock        0     2022.0   
4852139   tt1860375   tvSeries      30 Rock       30 Rock        0     1983.0   

         endYear  runtimeMinutes      genres  
476780    2013.0            22.0      Comedy  
4155396      NaN             NaN  Reality-TV  
4852139      NaN             NaN       Sport  


## Join: Story Names → IMDB Titles

In [5]:
# Limit IMDB to movies, TV series, and TV movies (most likely to appear in archetypometrics)
relevant_types = {'movie', 'tvSeries', 'tvMiniSeries', 'tvMovie', 'tvSpecial', 'short', 'tvShort'}
df_imdb_filtered = df_imdb[df_imdb['titleType'].isin(relevant_types)].copy()
df_imdb_filtered['title_lower'] = df_imdb_filtered['primaryTitle'].str.lower().str.strip()

# Build lookup: lowercase title → best IMDB row
# For duplicates, prefer the entry with the most data (non-null startYear)
df_imdb_filtered = df_imdb_filtered.sort_values('startYear', na_position='last')
imdb_lookup = df_imdb_filtered.drop_duplicates(subset='title_lower', keep='first').set_index('title_lower')

# Unique story names to match
unique_stories = df_chars['story_name'].unique()
story_lower = {s: s.lower().strip() for s in unique_stories}

# --- Pass 1: Exact match ---
exact_map = {}
for story, lower in story_lower.items():
    if lower in imdb_lookup.index:
        exact_map[story] = imdb_lookup.loc[lower]

print(f"Stories to match: {len(unique_stories)}")
print(f"Exact matches: {len(exact_map)}")

Stories to match: 341
Exact matches: 326


In [6]:
# --- Pass 2: Fuzzy match for unmatched stories ---
unmatched = [s for s in unique_stories if s not in exact_map]
# Filter out any NaN values from the IMDB index before fuzzy matching
all_imdb_titles_lower = [t for t in imdb_lookup.index if isinstance(t, str)]

fuzzy_map = {}
for story in unmatched:
    lower = story_lower[story]
    matches = get_close_matches(lower, all_imdb_titles_lower, n=1, cutoff=0.85)
    if matches:
        fuzzy_map[story] = imdb_lookup.loc[matches[0]]

print(f"Fuzzy matches: {len(fuzzy_map)}")
print(f"Total matched: {len(exact_map) + len(fuzzy_map)} / {len(unique_stories)}")
print(f"Unmatched stories: {[s for s in unique_stories if s not in exact_map and s not in fuzzy_map]}")

Fuzzy matches: 11
Total matched: 337 / 341
Unmatched stories: ['House, M.D.', 'Harry Potter', 'Firefly + Serenity', 'Calvin and Hobbes']


In [7]:
# --- Build story → IMDB mapping table ---
all_matches = {**exact_map, **fuzzy_map}
match_source = {s: 'exact' for s in exact_map}
match_source.update({s: 'fuzzy' for s in fuzzy_map})

imdb_cols = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'endYear', 'runtimeMinutes', 'genres']

story_imdb_rows = []
for story, row in all_matches.items():
    entry = {'story_name': story, 'match_type': match_source[story]}
    for col in imdb_cols:
        entry[f'imdb_{col}'] = row[col] if col in row.index else None
    story_imdb_rows.append(entry)

df_story_imdb = pd.DataFrame(story_imdb_rows)

# Left join characters to IMDB via story name
df_joined = df_chars.merge(df_story_imdb, on='story_name', how='left')

print(f"Joined rows: {len(df_joined):,}")
print(f"Characters with IMDB match: {df_joined['imdb_tconst'].notna().sum():,} / {len(df_joined):,}")
print(f"Match rate: {df_joined['imdb_tconst'].notna().mean():.1%}")

Joined rows: 2,000
Characters with IMDB match: 1,949 / 2,000
Match rate: 97.5%


## Joined Dataset Info

In [8]:
df_joined.info()
print()
df_joined.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   character_name        2000 non-null   object 
 1   character_story_pair  2000 non-null   object 
 2   story_name            2000 non-null   object 
 3   primary_archetype     2000 non-null   object 
 4   max_archetype_ratio   2000 non-null   float64
 5   match_type            1949 non-null   object 
 6   imdb_tconst           1949 non-null   object 
 7   imdb_titleType        1949 non-null   object 
 8   imdb_primaryTitle     1949 non-null   object 
 9   imdb_startYear        1949 non-null   float64
 10  imdb_endYear          720 non-null    float64
 11  imdb_runtimeMinutes   1662 non-null   float64
 12  imdb_genres           1909 non-null   object 
dtypes: float64(4), object(9)
memory usage: 203.3+ KB



,character_name,character_story_pair,story_name,primary_archetype,max_archetype_ratio,match_type,imdb_tconst,imdb_titleType,imdb_primaryTitle,imdb_startYear,imdb_endYear,imdb_runtimeMinutes,imdb_genres
0,Applejack,Applejack/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Angel-Brute-Hero,4.879310,exact,tt1751105,tvSeries,My Little Pony: Friendship Is Magic,2010.0,2019.0,22.0,"Adventure,Animation,Comedy"
1,Twilight Sparkle,Twilight Sparkle/My Little Pony: Friendship Is...,My Little Pony: Friendship Is Magic,Diva-Angel-Hero,10.590790,exact,tt1751105,tvSeries,My Little Pony: Friendship Is Magic,2010.0,2019.0,22.0,"Adventure,Animation,Comedy"
2,Rarity,Rarity/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Sophisticate-Diva-Hero,12.078822,exact,tt1751105,tvSeries,My Little Pony: Friendship Is Magic,2010.0,2019.0,22.0,"Adventure,Animation,Comedy"
3,Pinkie Pie,Pinkie Pie/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Adventurer,11.830696,exact,tt1751105,tvSeries,My Little Pony: Friendship Is Magic,2010.0,2019.0,22.0,"Adventure,Animation,Comedy"
4,Spike,Spike/My Little Pony: Friendship Is Magic,My Little Pony: Friendship Is Magic,Angel-Outcast-Adventurer,11.792687,exact,tt1751105,tvSeries,My Little Pony: Friendship Is Magic,2010.0,2019.0,22.0,"Adventure,Animation,Comedy"
5,Princess Celestia,Princess Celestia/My Little Pony: Friendship I...,My Little Pony: Friendship Is Magic,Sophisticate-Angel-Hero,23.942371,exact,tt1751105,tvSeries,My Little Pony: Friendship Is Magic,2010.0,2019.0,22.0,"Adventure,Animation,Comedy"
6,Leroy Jethro Gibbs,Leroy Jethro Gibbs/NCIS,NCIS,Hero,7.959240,exact,tt0364845,tvSeries,NCIS,2003.0,NaN,60.0,"Action,Crime,Drama"
7,Anthony DiNozzo,Anthony DiNozzo/NCIS,NCIS,Hero-Brute-Adventurer,8.321907,exact,tt0364845,tvSeries,NCIS,2003.0,NaN,60.0,"Action,Crime,Drama"
8,Abby Sciuto,Abby Sciuto/NCIS,NCIS,Geek-Hero-Adventurer,9.163252,exact,tt0364845,tvSeries,NCIS,2003.0,NaN,60.0,"Action,Crime,Drama"
9,Donald Mallard,Donald Mallard/NCIS,NCIS,Angel-Hero,10.309380,exact,tt0364845,tvSeries,NCIS,2003.0,NaN,60.0,"Action,Crime,Drama"


In [53]:
# print("Sample of joined data — characters with their IMDB metadata:")
# df_joined[df_joined['imdb_tconst'].notna()][
#     ['character_name', 'story_name', 'primary_archetype', 'imdb_titleType', 'imdb_startYear', 'imdb_genres', 'match_type']
# ].head(20)

# print(df_joined['imdb_genres'].value_counts())

# # create new df from df_joined where we group by story_name
# df_grouped = df_joined.groupby('story_name').agg({
#     'character_name': 'count',
#     'primary_archetype': 'nunique',
#     'imdb_genres': lambda x: x.mode()[0] if not x.mode().empty else None,
    
# }).reset_index()
# print(df_grouped.head())

# df_grouped.to_csv('story_character_summary.csv', index=False)
imdb_genres = df_joined['imdb_genres'].str.get_dummies(sep=',')
df_joined = pd.concat([df_joined, imdb_genres], axis=1)

drama_cols = [col for col in df_joined.columns if 'Drama' in col]
comedy_cols = [col for col in df_joined.columns if 'Comedy' in col]

# Add new column, is_drama_or_comedy, which is 1 if the story is in either the Drama or Comedy genre, but not both, and 0 otherwise
df_joined['is_drama_or_comedy'] = df_joined[drama_cols].any(axis=1) ^ df_joined[comedy_cols].any(axis=1)
df_joined['is_drama_or_comedy'] = df_joined['is_drama_or_comedy'].apply(lambda x: 1 if x else 0)


In [54]:
df_joined['is_drama_or_comedy'].value_counts()

is_drama_or_comedy
1    1180
0     820
Name: count, dtype: int64

In [25]:
# show all rows in pandas df
pd.set_option('display.max_rows', None)

In [26]:
df_joined[df_joined['match_type'] == 'fuzzy']

,character_name,character_story_pair,story_name,primary_archetype,max_archetype_ratio,match_type,imdb_tconst,imdb_titleType,imdb_primaryTitle,imdb_startYear,imdb_endYear,imdb_runtimeMinutes,imdb_genres
363,Huckleberry Finn,Huckleberry Finn/Adventures of Huckleberry Finn,Adventures of Huckleberry Finn,Adventurer,7.332084,fuzzy,tt0031020,movie,The Adventures of Huckleberry Finn,1939.0,NaN,91.0,"Adventure,Drama,Family"
364,Jim,Jim/Adventures of Huckleberry Finn,Adventures of Huckleberry Finn,Lone Wolf-Hero-Angel,7.549585,fuzzy,tt0031020,movie,The Adventures of Huckleberry Finn,1939.0,NaN,91.0,"Adventure,Drama,Family"
365,Olivia Benson,Olivia Benson/Law & Order: SVU,Law & Order: SVU,Hero,11.990900,fuzzy,tt1166893,tvSeries,Law & Order: UK,2009.0,2014.0,46.0,"Action,Adventure,Crime"
366,John Munch,John Munch/Law & Order: SVU,Law & Order: SVU,Hero,9.315851,fuzzy,tt1166893,tvSeries,Law & Order: UK,2009.0,2014.0,46.0,"Action,Adventure,Crime"
367,Donald Cragen,Donald Cragen/Law & Order: SVU,Law & Order: SVU,Traditionalist-Hero,11.892449,fuzzy,tt1166893,tvSeries,Law & Order: UK,2009.0,2014.0,46.0,"Action,Adventure,Crime"
368,Elliot Stabler,Elliot Stabler/Law & Order: SVU,Law & Order: SVU,Brute-Hero,8.347661,fuzzy,tt1166893,tvSeries,Law & Order: UK,2009.0,2014.0,46.0,"Action,Adventure,Crime"
369,Odafin Tutuola,Odafin Tutuola/Law & Order: SVU,Law & Order: SVU,Lone Wolf-Hero,6.099246,fuzzy,tt1166893,tvSeries,Law & Order: UK,2009.0,2014.0,46.0,"Action,Adventure,Crime"
370,Melinda Warner,Melinda Warner/Law & Order: SVU,Law & Order: SVU,Angel-Hero,6.750318,fuzzy,tt1166893,tvSeries,Law & Order: UK,2009.0,2014.0,46.0,"Action,Adventure,Crime"
428,Eric Forman,Eric Forman/That 70's Show,That 70's Show,Angel-Diva-Outcast,7.873467,fuzzy,tt0165598,tvSeries,That '70s Show,1998.0,2006.0,22.0,"Comedy,Drama,Romance"
429,Jackie Burkhart,Jackie Burkhart/That 70's Show,That 70's Show,Demon-Adventurer-Diva,17.340115,fuzzy,tt0165598,tvSeries,That '70s Show,1998.0,2006.0,22.0,"Comedy,Drama,Romance"
